In [1]:
# install first:  pip install pypdf

from pypdf import PdfReader

def extract_pdf_text(filepath):
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        # TODO: each `page` has a .extract_text() method that returns that page's text.
        # append it to `text` (add a space or newline between pages so words don't merge)
        extracted_text=page.extract_text()
        
        text+=extracted_text+"\n"
        
    return text

full_text = extract_pdf_text("Mukhlif_Aljaberi_CV (2).pdf")
import re
full_text = re.sub(r'\s+', ' ', full_text)  # collapses any run of whitespace into a single space
print(f"Total characters: {len(full_text)}")
print(full_text[:500])   # sanity check — look at the first 500 characters

Total characters: 2037
Computer Engineering graduate specialising in Artificial Intelligence, combining strong academic foundations in deep learning, computer vision and generative AI with hands-on experience delivering full-stack AI-powered web applications. Proven ability to research and implement cutting-edge models (GANs, diffusion, face-swap) within production pipelines. Eager to contribute to innovative AI teams and continue growing as an engineer. Mukhlif Mohammed Aljaberi AI & Computer Engineering Graduate mok


In [2]:
def chunk_text_from_string(text, chunk_size=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = words[i:i+chunk_size]
        x = ' '.join(chunk)
        chunks.append(x)
    return chunks

chunks = chunk_text_from_string(full_text)
print(f"Got {len(chunks)} chunks")

Got 2 chunks


In [3]:
# a new collection for the CV-vs-job tool
import chromadb

# creates a persistent database saved to disk in a folder called "chroma_db"
chroma_client = chromadb.PersistentClient(path="./chroma_db_cv_job")
cv_job_collection = chroma_client.get_or_create_collection(name="cv_job_matcher")

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # small, free, good enough

def embed_chunks(chunks):
    embeddings = []
    for chunk in chunks:
        # TODO: use `model.encode(chunk)` to get the embedding for this one chunk,
        # and append it to `embeddings`
        embidding=model.encode(chunk)
        embeddings.append(embidding)
        pass
    return embeddings

embeddings = embed_chunks(chunks)
print(len(embeddings))        # should match number of chunks
print(len(embeddings[0]))     # this is the vector length — what number do you get?

c:\Users\Lenovo\Desktop\RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8605.67it/s]


2
384


In [5]:
def add_document(text, source_label, collection):
    # clean + chunk (reuse what you already have)
    import re
    text = re.sub(r'\s+', ' ', text)
    doc_chunks = chunk_text_from_string(text, chunk_size=150)
    
    # embed
    doc_embeddings = [emb.tolist() for emb in embed_chunks(doc_chunks)]
    
    # unique ids — must be unique across BOTH documents, so prefix with the source
    ids = [f"{source_label}_chunk_{i}" for i in range(len(doc_chunks))]
    
    # TODO: build the metadata list — one dict per chunk, each like {"source": source_label}
    # you need len(doc_chunks) of them, all with the same source_label
    metadatas =[{"source":source_label} for _ in range(len(doc_chunks))]
    
    collection.add(
        documents=doc_chunks,
        embeddings=doc_embeddings,
        ids=ids,
        metadatas=metadatas
    )
    print(f"Added {len(doc_chunks)} chunks from '{source_label}'")

In [6]:
# CV — from PDF (reuse your extractor)
cv_text = extract_pdf_text("Mukhlif_Aljaberi_CV (2).pdf")

# Job description — paste a real one as a string
job_text = """Page 1 of 3
Job Description
Summary
Job Title: Analyst / Econometrician or Data Scientist
Reporting to: Managing Director
Location: Offshore office 5 days a week in the office
Type of contract: Full-time
Hours: Monday-Friday, 9:30am-5:30pm (GMT+3) (Covering UK Core Hours).
Please send your CV to: Khaled.Ekrayem@Adclaro.com
About the role
Due to rapid and growing demand for analytics services, we are expanding our offshore capability and launching a new team. This role begins with a paid three‑month training programme designed to ensure a smooth transition and strong alignment with our key UK client team. You will start contributing to real project work from the second month of the programme. The first six months of employment form part of the probation period, after which successful candidates will move into full‑time roles within the core offshore team and specialise as an Econometrician or Data Scientist.
We are seeking a highly motivated and capable Analyst with experience and familiarity in data analysis and processing, with skills in Excel/BI, Python or R along with Power Point.
As such, the candidate will support the existing and new projects by:
•
Strong communication skills in English (written and verbal), with the confidence to collaborate effectively with the UK analytics team.
•
Proficiency in SQL, R or Python, with hands‑on experience writing and debugging code.
•
Experience in data processing and preparation, including cleaning, validating, and transforming large datasets using Python, SQL, Excel, and/or Power BI.
•
Ability to run and maintain automated workflows and scripts, ensuring data pipelines are accurate, efficient, and well‑documented.
•
Ownership of end‑to‑end data preparation, including identifying anomalies, proactively flagging data quality issues, and understanding the modelling requirements to resolve them.
•
Support for the full MMM modelling lifecycle, from data exploration and variable selection through to model building, diagnostics, interpretation, and preparation of insights.
•
Support in producing clear, structured, and insightful deliverables, including internal analysis summaries and client‑ready presentation decks.
•
Comfort working in a backend support capacity, ensuring timely, reliable delivery of all data and modelling tasks for the UK analytics team.
This role is ideal for candidates who are eager to learn and develop in the analytics space and are comfortable working with complex datasets and issues under time pressure. This is an opportunity to take your career to the next level and gain hands-on experience. The ideal candidate will be motivated and supported to grow beyond technical expertise and build the leadership and communication capabilities needed for long-term career progression.
Page 2 of 3
Candidate profile
The successful candidate will be expected to grow into the role taking ownership of raw data ingestion, processing and modelling with minimal supervision.
This includes strong skills for gathering, checking and processing data in preparation for analysis and being comfortable taking models from start to finish independently. There will be opportunity to learn designing meaningful reports that effectively link project scope, client expectations and actionable recommendations.
Technical proficiency
•
Experience with statistical or econometric modelling (e.g., regression, regularisation techniques, model validation) using tools such as Python, R, or other statistical software packages (e.g., EViews, SAS).
•
Experience with model validation, diagnostics and performance metrics.
•
Hands-on experience with Machine Learning, Bayesian Techniques is highly valued.
•
Familiarity with visualisation tools (Power BI, Tableau or similar).
•
Ability to handle large datasets, clean and transform raw data, and drive insight.
Nice to have:
•
Experience with Python libraries such as Pandas and Parquet is a big plus.
•
Experience with R libraries such as Tidyverse is a big plus.
•
Experience with cloud environments (Azure, AWS, GCP) or Git‑based version control.
•
Beginner level understanding of marketing and media metrics, business KPIs, and MMM concepts.
Problem solving & communication
•
Clear communicator in English (written and verbal) is a must.
•
Strong analytical thinking, problem-solving, and attention to detail.
•
Comfortable working in a collaborative environment.
•
Ability to manage multiple tasks and deliver to deadlines in a fast‑paced environment.
Page 3 of 3
About us
Zafir Insight is the offshore office for supporting international analytics teams across the UK/EEA in delivering Data Science consulting projects with specific expertise in MMM. Our main client is Adclaro in the UK, an established and fast-growing marketing effectiveness agency dedicated to providing best-in-class marketing mix modelling (MMM) services to a range of high-profile brands. Our work helps Agencies understand their media and marketing performance in terms of what has worked in the past, and how to optimise the future.
Our work arrangements give us a unique insight into how our clients operate and help us remain close to our clients' needs and maintain relationships for long periods of time.
We place a strong emphasis on openness and transparency when it comes to our services and work closely with our clients to both verify and embed the results of our work. We learn, develop and succeed together, fostering an ego-free and positive company culture, to ensure we can achieve business goals and personal dreams at the same time.
We are constantly striving to improve the efficiency and technical capability of our modelling processes, enabling us to support the delivery of best-in-class analytics to our clients.
Diversity, Equity & Inclusion at Adclaro
We are an equal opportunity employer and are committed to building an inclusive and diverse workforce. We welcome applicants from all backgrounds, experiences and perspectives, striving to create a workplace where everyone can thrive and contribute their best.
Here’s how we support you
•
Best in class paid 3-months training programme to launch your analytics career.
•
Be part of cutting‑edge research, project work, hands‑on analytics, and continuous learning.
•
An opportunity to shape your career path and evolve the role as the company grows.
•
Competitive salary.
•
Opportunity to join a rapidly growing business.
•
Annual bonus based on individual and company performance.
•
15 Days plus 7 flexible Eid days and 3 public holidays at Christmas.
•
Additional holiday days for long-term employees to recognize loyalty and commitment.
•
Regular company events & team socials to bring everyone together.
•
Free tea, coffee, and snacks, and fruit around the office.
What to expect next?
Should you see a strong fit, we can’t wait to learn more about you as a person and the experience you bring. Please send your CV to: Khaled.Ekrayem@Adclaro.com
After you submit your application, the next steps, if successful, will include three main stages:
1
- Initial screening call followed by screening interview.
2
- Analytics task is sent for you to complete and present in the second interview.
3
- You will find out the outcome after the second interview.
At Zafir Insight we respect your privacy and are committed to protecting the personal data you provide during recruitment. Thank you for considering a career with Zafir Insight.
[paste the full text of a real job posting here — 
title, responsibilities, requirements, everything]
"""

# load both into the collection with their source tags
add_document(cv_text, "cv", cv_job_collection)
add_document(job_text, "job", cv_job_collection)

print("Total chunks in collection:", cv_job_collection.count())

Added 2 chunks from 'cv'
Added 8 chunks from 'job'
Total chunks in collection: 10


In [7]:
def retrieve_for_matching(query, collection, top_k=3):
    query_embedding = model.encode(query).tolist()
    
    # retrieve top chunks ONLY from the CV
    cv_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"source": "cv"}      # <-- the metadata filter doing the work
    )
    
    # retrieve top chunks ONLY from the job description
    job_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"source": "job"}
    )
    
    cv_chunks = cv_results['documents'][0]
    job_chunks = job_results['documents'][0]
    return cv_chunks, job_chunks
    
cv_chunks, job_chunks = retrieve_for_matching("what technical skills are required?", cv_job_collection)
print("FROM CV:", cv_chunks)
print("\nFROM JOB:", job_chunks)

FROM CV: ['Computer Engineering graduate specialising in Artificial Intelligence, combining strong academic foundations in deep learning, computer vision and generative AI with hands-on experience delivering full-stack AI-powered web applications. Proven ability to research and implement cutting-edge models (GANs, diffusion, face-swap) within production pipelines. Eager to contribute to innovative AI teams and continue growing as an engineer. Mukhlif Mohammed Aljaberi AI & Computer Engineering Graduate mokhlifaljabri2001@gmail.com +963 986 961 587 EDUCATION B.Sc. Computer Engineering — Specialisation in Artificial Intelligence Arab International University Graduation Year: 2026 GPA: 2.7 / 4.0 TECHNICAL SKILLS AI / ML / DL Machine Learning Deep Learning GANs Computer Vision PyTorch TensorFlow AI MODELS & TOOLS YOLO / YOLOe InsightFace SDXL TinyLlama In-painting Stable Diffusion LANGUAGES Python SQL HTML / CSS JavaScript FRAMEWORKS Laravel React Bootstrap ASP .NET ACADEMIC PROJECTS AI Fu

In [9]:
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
client = Groq()  # now it can find the key
from groq import Groq

client = Groq() 

def analyze_match(cv_chunks, job_chunks):
    cv_context = "\n".join(cv_chunks)
    job_context = "\n".join(job_chunks)
    
    prompt = f"""You are a career advisor. Compare the candidate's CV against the job requirements below and give an honest assessment.

CANDIDATE'S CV (relevant parts):
{cv_context}

JOB REQUIREMENTS (relevant parts):
{job_context}

Provide:
1. MATCH: What does the candidate genuinely have that this role needs?
2. GAPS: What key requirements does the candidate NOT appear to meet?
3. HONEST VERDICT: Is this a strong fit, a stretch, or a mismatch? Be direct, not encouraging for its own sake.

Base your answer ONLY on what's in the CV and job text above."""
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=700
    )
    return response.choices[0].message.content

# run the full thing
query = "what are the key skills and requirements for this role?"
cv_chunks, job_chunks = retrieve_for_matching(query, cv_job_collection)
print(analyze_match(cv_chunks, job_chunks))

**MATCH:**
The candidate has some technical skills that align with the job requirements, such as:
- Experience with Machine Learning (listed as a technical skill)
- Proficiency in Python (a required technical skill)
- Familiarity with data processing and modeling (demonstrated through academic projects)

**GAPS:**
The candidate appears to lack:
- Direct experience with statistical or econometric modeling (e.g., regression, regularization techniques, model validation)
- Experience with specific tools like R, EViews, SAS, or cloud environments (Azure, AWS, GCP)
- Hands-on experience with data visualization tools like Power BI or Tableau
- Explicit experience with data ingestion, processing, and modeling in a professional setting (most experience listed is from academic projects)

**HONEST VERDICT:**
This is a mismatch. While the candidate has a strong foundation in AI, Machine Learning, and programming, their experience and skills do not directly align with the specific requirements of t

In [10]:
def analyze_cv_vs_job(cv_file, job_text):
    # 1. extract CV text from the uploaded PDF
    cv_text = extract_pdf_text(cv_file)
    
    # 2. fresh collection each run, so old data doesn't pile up
    #    (delete + recreate to keep it clean)
    try:
        chroma_client.delete_collection(name="cv_job_matcher")
    except:
        pass
    collection = chroma_client.get_or_create_collection(name="cv_job_matcher")
    
    # 3. load both documents with source tags
    add_document(cv_text, "cv", collection)
    add_document(job_text, "job", collection)
    
    # 4. retrieve + analyze
    query = "what are the key skills and requirements for this role?"
    cv_chunks, job_chunks = retrieve_for_matching(query, collection)
    result = analyze_match(cv_chunks, job_chunks)
    
    return result

In [11]:
analyze_cv_vs_job("Mukhlif_Aljaberi_CV (2).pdf",job_text)

Added 2 chunks from 'cv'
Added 8 chunks from 'job'


"**1. MATCH:** \nThe candidate has experience with Machine Learning, which is highly valued in the job requirements. They also have hands-on experience with Python, which is one of the tools mentioned in the job requirements. Additionally, the candidate has worked with large datasets and has experience with data processing and modeling, which are essential skills for the role.\n\n**2. GAPS:** \nThe candidate lacks direct experience with statistical or econometric modeling, which is a key requirement for the role. They also do not mention experience with model validation, diagnostics, and performance metrics, which are crucial skills for the position. Furthermore, the candidate does not appear to have experience with data visualization tools like Power BI or Tableau, or with cloud environments and Git-based version control, which are nice-to-have skills. The candidate's background is more focused on Computer Vision and Generative AI, which may not be directly applicable to the role.\n\n

In [12]:
import gradio as gr

def gradio_analyze(cv_file, job_text):
    if cv_file is None:
        return "Please upload a CV (PDF)."
    if not job_text or job_text.strip() == "":
        return "Please paste a job description."
    return analyze_cv_vs_job(cv_file, job_text)

demo = gr.Interface(
    fn=gradio_analyze,
    inputs=[
        gr.File(label="Upload your CV (PDF)", file_types=[".pdf"], type="filepath"),
        gr.Textbox(label="Paste the job description", lines=12, placeholder="Paste the full job posting here...")
    ],
    outputs=gr.Markdown(label="Match Analysis"),
    title="CV ↔ Job Match Analyzer",
    description="Upload your CV and paste a job description. Get an honest assessment of your fit, your strengths, and your gaps — powered by RAG."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Lenovo\Desktop\RAG\.venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Added 2 chunks from 'cv'
Added 8 chunks from 'job'


In [ ]:
!pip install gradio

In [ ]:
print("ehloo")